# EDA датасета многоплоскостного ДОЭ

Прикрепите `ai_phonetics_data_v1` как Kaggle Input. Notebook проверяет состав манифестов, различия train/val, разреженность и частотный состав целей, а также похожесть четырёх картинок внутри одной четвёрки. Результаты сохраняются в `/kaggle/working/eda/`.

Это EDA данных, а не выбор новой архитектуры.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont

KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v1')
EDA_QUADS = {'train': 2_048, 'val': 2_000, 'test_internal': 2_000}  # None = весь split
OUTPUT_DIR = Path('/kaggle/working/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_root(path):
    if (path / 'dataset_metadata.json').is_file(): return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1: raise FileNotFoundError(f'Cannot find dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_root(KAGGLE_DATASET_DIR)
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
print('DATA_ROOT =', DATA_ROOT)


## Загрузчик для EDA

Код повторяет правила baseline: grayscale, crop `128×128`, синтетика по seed и нормировка каждого канала до максимума 1.

In [ ]:
def normalize(x):
    maximum = x.max()
    if maximum <= 1e-8: raise ValueError('All-black target')
    return x / maximum

def tensor_from_pil(image):
    return normalize(torch.from_numpy(np.asarray(image.convert('L'), dtype=np.float32) / 255.0))

def draw_glyph(canvas, rng, value):
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]; size = int(rng.integers(22, 66))
    try: font = ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError: font = ImageFont.load_default()
    glyph = Image.new('L', (size * 2, size * 2), 0); draw = ImageDraw.Draw(glyph); box = draw.textbbox((0, 0), char, font=font)
    draw.text((-box[0], -box[1]), char, fill=value, font=font); box = glyph.getbbox()
    if box is not None:
        glyph = glyph.crop(box).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
        canvas.paste(glyph, (int(rng.integers(0, max(1, 129 - glyph.width))), int(rng.integers(0, max(1, 129 - glyph.height)))), glyph)

def synthetic(seed):
    rng = np.random.default_rng(int(seed)); canvas = Image.new('L', (128, 128), 0); draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph'])); value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, 124, size=2).tolist()); y0, y1 = sorted(rng.integers(4, 124, size=2).tolist()); x1, y1 = min(127, max(x1, x0 + 8)), min(127, max(y1, y0 + 8))
        if kind == 'ellipse': draw.ellipse((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'rectangle': draw.rectangle((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'line': draw.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring': draw.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            r = int(rng.integers(2, 10)); cx, cy = int(rng.integers(r, 128-r)), int(rng.integers(r, 128-r)); draw.ellipse((cx-r, cy-r, cx+r, cy+r), fill=value)
        else: draw_glyph(canvas, rng, value)
    return tensor_from_pil(canvas)

def load_item(item):
    if item['source'] == 'synthetic': return synthetic(item['seed'])
    path = BSDS_ROOT / item['path']; x, y = item['crop']['x'], item['crop']['y']
    with Image.open(path) as image: return tensor_from_pil(image.crop((x, y, x + 128, y + 128)))

def read_manifest(split):
    return [json.loads(line) for line in (DATA_ROOT / 'manifests' / f'{split}.jsonl').read_text(encoding='utf-8').splitlines() if line]

manifests = {split: read_manifest(split) for split in EDA_QUADS}
print({name: len(records) for name, records in manifests.items()})


## Состав манифестов и визуальная проверка

In [ ]:
def manifest_summary(records):
    items = [item for record in records for item in record['items']]
    natural = [item for item in items if item['source'] == 'bsds']
    files = pd.Series([item['path'] for item in natural])
    return {'quads': len(records), 'targets': len(items), 'synthetic_share': round(1 - len(natural) / len(items), 4), 'natural_share': round(len(natural) / len(items), 4), 'unique_natural_files': int(files.nunique()) if len(files) else 0, 'max_file_reuse': int(files.value_counts().max()) if len(files) else 0}

summary = pd.DataFrame({name: manifest_summary(records) for name, records in manifests.items()}).T
display(summary)
summary.to_csv(OUTPUT_DIR / 'manifest_summary.csv')

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for row, record in enumerate(manifests['train'][:3]):
    for col, item in enumerate(record['items']):
        axes[row, col].imshow(load_item(item), cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(item['source']); axes[row, col].axis('off')
fig.suptitle('Три случайных train-четвёрки: источник каждого канала')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'quad_samples.png', dpi=160, bbox_inches='tight')
plt.show()


## Распределения изображений, спектров и похожести внутри quad

In [ ]:
yy, xx = np.ogrid[-64:64, -64:64]; radius = np.sqrt(xx**2 + yy**2); high_mask = radius >= 32

def pearson(a, b):
    a, b = a.ravel(), b.ravel(); a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (np.sqrt((a*a).sum() * (b*b).sum()) + 1e-8))

def image_features(image):
    a = image.numpy(); hist, _ = np.histogram(a, bins=32, range=(0, 1), density=True); p = hist / (hist.sum() + 1e-12); spectrum = np.abs(np.fft.fftshift(np.fft.fft2(a)))**2
    return {'mean': float(a.mean()), 'std': float(a.std()), 'nonzero_fraction': float((a > 1e-3).mean()), 'entropy': float(-(p[p > 0] * np.log2(p[p > 0])).sum()), 'high_frequency_energy': float(spectrum[high_mask].sum() / (spectrum.sum() + 1e-12))}

def analyse(split, records, limit):
    indices = np.linspace(0, len(records) - 1, min(limit or len(records), len(records)), dtype=int)
    rows, pair_cc = [], []
    for index in indices:
        record = records[int(index)]; images = [load_item(item) for item in record['items']]
        arrays = [image.numpy() for image in images]
        for channel, (item, image) in enumerate(zip(record['items'], images)):
            row = image_features(image); row.update({'split': split, 'source': item['source'], 'quad_index': int(index), 'channel': channel}); rows.append(row)
        for left in range(4):
            for right in range(left + 1, 4): pair_cc.append({'split': split, 'pair_cc': pearson(arrays[left], arrays[right])})
    return pd.DataFrame(rows), pd.DataFrame(pair_cc)

feature_frames, correlation_frames = [], []
for split, records in manifests.items():
    features, correlations = analyse(split, records, EDA_QUADS[split]); feature_frames.append(features); correlation_frames.append(correlations)
features = pd.concat(feature_frames, ignore_index=True); correlations = pd.concat(correlation_frames, ignore_index=True)
features.to_csv(OUTPUT_DIR / 'image_features.csv', index=False); correlations.to_csv(OUTPUT_DIR / 'quad_pair_correlations.csv', index=False)
display(features.groupby(['split', 'source'])[['mean', 'std', 'nonzero_fraction', 'entropy', 'high_frequency_energy']].agg(['mean', 'std']).round(4))
display(correlations.groupby('split')['pair_cc'].agg(['mean', 'std', 'min', 'max']).round(4))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for axis, column in zip(axes.flat, ['mean', 'std', 'nonzero_fraction', 'entropy', 'high_frequency_energy']):
    for (split, source), frame in features.groupby(['split', 'source']): axis.hist(frame[column], bins=35, density=True, alpha=.35, label=f'{split}/{source}')
    axis.set_title(column); axis.legend(fontsize=7)
axes.flat[-1].hist(correlations['pair_cc'], bins=40, color='slateblue'); axes.flat[-1].set_title('Pearson внутри одной четвёрки')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'distributions.png', dpi=160, bbox_inches='tight'); plt.show()

report = {'manifest_summary': summary.to_dict(orient='index'), 'feature_means': features.groupby(['split', 'source'])[['mean', 'std', 'nonzero_fraction', 'entropy', 'high_frequency_energy']].mean().round(6).to_dict(orient='index'), 'pair_cc': correlations.groupby('split')['pair_cc'].agg(['mean', 'std', 'min', 'max']).round(6).to_dict(orient='index')}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('EDA artefacts saved to', OUTPUT_DIR)
